# B01：第一个完整建模任务（辅修学生练习）

2 实验学时，T02 后。开工前用计划产量预测完工后的总能耗。数据为独立合成批次。

本练习仅划分训练与验证，**没有独立最终测试集**。`test_size` 只是库参数名，这里较小部分用于验证比较。

先把整个单元复制到非保护盘个人目录，按顺序完成 TODO。完成每处 TODO 后，删除该处 `raise NotImplementedError(...)` 停止提示，保留后续检查代码；最后重启内核全运行。线性回归原理留到 T03。

## 1. 读取与检查（已提供）

原始表完整，无需填补或删行。计划产量在开工前确定；能耗和最终电费在完工后产生。编号用于核对记录，电费或标签不能放进输入。

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error

candidates = []
for parent in (Path.cwd(), *Path.cwd().parents):
    candidates.extend(
        [
            parent / "data/b01_batches.csv",
            parent / "units/split-evaluation/data/b01_batches.csv",
        ]
    )
data_path = next((p for p in candidates if p.is_file()), None)
if data_path is None:
    raise FileNotFoundError("请保留单元目录关系，并从课程目录启动Jupyter")
raw = pd.read_csv(data_path)
display(raw.head())
print("原始形状：", raw.shape)
print("缺失数：", raw.isna().sum().to_dict())
assert raw.shape == (120, 4) and raw.batch_id.is_unique
assert not raw.isna().any().any()

## 2. 形成输入与标签

X 保留二维特征轴，y 使用一维；编号保持相同行顺序。代码已提供，运行后解释形状。

In [ ]:
feature_cols = ["planned_units"]
X = raw[feature_cols].to_numpy(dtype=float)
y = raw["energy_kwh"].to_numpy(dtype=float)
ids = raw["batch_id"].to_numpy()
print("X:", X.shape, "y:", y.shape)

In [ ]:
assert feature_cols == ["planned_units"]
assert X.shape == (120, 1) and y.shape == (120,) and ids.shape == (120,)
np.testing.assert_array_equal(X[:, 0], raw["planned_units"].to_numpy())
np.testing.assert_array_equal(y, raw["energy_kwh"].to_numpy())
np.testing.assert_array_equal(ids, raw["batch_id"].to_numpy())

## 3. 一起划分，保持对应

回看讲义：T02 第 8.2 节，数据划分与行对应。

调用 `train_test_split(X, y, ids, test_size=1 / 3, random_state=42)`。依次接收 X 两部分、y 两部分、ids 两部分，以 train/val 命名。固定种子用于复现，不用于挑选好分数。

本次假设批次独立且来自稳定过程，采用随机划分；重复设备或未来预测任务不能照搬。

In [ ]:
# TODO：把 ... 替换为一起划分的三个数组；完成后删除停止提示。
raise NotImplementedError("请补全本格")
X_train, X_val, y_train, y_val, ids_train, ids_val = train_test_split(
    ..., ..., ..., test_size=1 / 3, random_state=42
)
print("训练：", len(y_train), "验证：", len(y_val))

In [ ]:
assert X_train.shape == (80, 1) and X_val.shape == (40, 1)
assert y_train.shape == (80,) and y_val.shape == (40,)
assert set(ids_train).isdisjoint(set(ids_val))
assert set(ids_train) | set(ids_val) == set(ids)
for sample_ids, xx, yy in [(ids_train, X_train, y_train), (ids_val, X_val, y_val)]:
    ordered = raw.set_index("batch_id").loc[sample_ids]
    np.testing.assert_array_equal(xx[:, 0], ordered["planned_units"].to_numpy())
    np.testing.assert_array_equal(yy, ordered["energy_kwh"].to_numpy())
# 检查格已提供：核对本实验规定的种子和划分比例，不挑选更好分数。
_expected_train_ids, _expected_val_ids = train_test_split(
    ids, test_size=1 / 3, random_state=42
)
np.testing.assert_array_equal(
    ids_train,
    _expected_train_ids,
    err_msg="训练编号不符合固定划分：test_size=1/3、random_state=42",
)
np.testing.assert_array_equal(
    ids_val,
    _expected_val_ids,
    err_msg="验证编号不符合固定划分：test_size=1/3、random_state=42",
)
print("固定划分与行对应检查通过")

## 4. 训练、预测与同条件比较

回看讲义：T02 第 8.3 节，训练、预测与评价接口。

`fit(X_train, y_train)` 只从训练部分学习；`predict(X_val)` 生成验证预测；指标函数比较真实值和对应预测。

`DummyRegressor(strategy="mean")` 学习训练标签均值；`LinearRegression()` 提供线性模型。MAE 用 `mean_absolute_error`；RMSE 用 `np.sqrt(mean_squared_error(...))`。不使用默认 `.score()`，不调参。

In [ ]:
# TODO：补全两次fit的训练输入和标签，完成后删除停止提示。
raise NotImplementedError("请补全本格")
baseline = DummyRegressor(strategy="mean")
model = LinearRegression()
baseline.fit(..., ...)
model.fit(..., ...)
baseline_train = baseline.predict(X_train)
baseline_val = baseline.predict(X_val)
linear_train = model.predict(X_train)
linear_val = model.predict(X_val)

下方外层汇总已提供；完成返回字典的两个指标。yy 和 pred 必须来自同一数据部分。

In [ ]:
def metric_row(name, split_name, yy, pred):
    # TODO：补全两项指标表达式，再删除停止提示。
    raise NotImplementedError("请补全指标")
    return {"model": name, "split": split_name, "MAE": ..., "RMSE": ...}

In [ ]:
results = pd.DataFrame(
    [
        metric_row("Dummy", "train", y_train, baseline_train),
        metric_row("Dummy", "validation", y_val, baseline_val),
        metric_row("Linear", "train", y_train, linear_train),
        metric_row("Linear", "validation", y_val, linear_val),
    ]
)
display(results.round(3))

In [ ]:
assert set(results.columns) == {"model", "split", "MAE", "RMSE"}
assert len(results) == 4 and not results.duplicated(["model", "split"]).any()
assert set(zip(results.model, results.split)) == {
    ("Dummy", "train"),
    ("Dummy", "validation"),
    ("Linear", "train"),
    ("Linear", "validation"),
}
np.testing.assert_allclose(baseline.predict(X_val), np.mean(y_train))
assert np.isfinite(results[["MAE", "RMSE"]].to_numpy()).all()
assert (results[["MAE", "RMSE"]] >= 0).all().all()
assert (results.RMSE >= results.MAE - 1e-10).all()
for name, fitted in [("Dummy", baseline), ("Linear", model)]:
    for label, xx, yy in [("train", X_train, y_train), ("validation", X_val, y_val)]:
        r = results[(results.model == name) & (results.split == label)].iloc[0]
        pred = fitted.predict(xx)
        np.testing.assert_allclose(r.MAE, np.mean(np.abs(pred - yy)))
        np.testing.assert_allclose(r.RMSE, np.sqrt(np.mean((pred - yy) ** 2)))
# 另建参照仅用于检查，不是第三个候选，也不会重新训练你的 model。
# 先确认你的预测与“只用训练集拟合”得到的预测一致。
_training_check = LinearRegression().fit(X_train, y_train)
for _check_X in (X_train, X_val):
    np.testing.assert_allclose(
        model.predict(_check_X),
        _training_check.predict(_check_X),
        rtol=1e-9,
        atol=1e-9,
        err_msg="线性预测与仅使用训练集的参照不一致，请检查fit的数据范围",
    )
print("基线来源、线性训练范围与指标检查通过；仍需本人解释数据用途")

## 5. 看图与解释（绘图已提供）

显示验证 MAE，单位 kWh，不是最终测试结果。

In [ ]:
validation = results[results.split == "validation"].set_index("model")
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.bar(validation.index, validation.MAE, color=["#8997a5", "#157a83"])
ax.set_ylabel("Validation MAE (kWh)")
ax.set_title("Same validation samples, same metric")
fig.tight_layout()
plt.show()

回答：

1. 基线均值来自哪里，为什么不用全表均值？
2. 哪种方法的验证 MAE 较小？引用实际数值。
3. 为什么训练分数和这次验证结果都不能称为独立最终测试表现？

**我的回答：**（填写三项短答）

## 6. 保存与个人核验

重启内核并运行全部，保存到非保护盘并按课程渠道提交或另留副本。只提交本 Notebook，不另交长报告。

指出 fit 和 predict 分别用哪部分数据；解释为何不能加入最终电费，即使它让分数变好。检查格不能代替个人解释。